In [2]:
import pandas as pd

df = pd.read_csv('/content/FantasyPros_Fantasy_Football_Projections_FLX.csv')

df.head()

,Player,Pos,Team,FantasyPoints
0,Jalen Hurts,QB,PHI,351.8
1,Josh Allen,QB,BUF,349.6
2,Patrick Mahomes,QB,KC,328.7
3,Lamar Jackson,QB,BAL,327.3
4,Christian McCaffrey,RB,SF,315.3


In [3]:
adp_df = pd.read_csv('/content/FantasyPros_2024_Overall_ADP_Rankings.csv')

adp_df['ADP Rank'] = adp_df['Current ADP'].rank()

adp_df_cutoff = adp_df[:100]

adp_df.head()

,Rank,Player,Team,Pos,Current ADP,ADP Rank
0,1,Christian McCaffrey,SF,RB,1,1.0
1,2,CeeDee Lamb,DAL,WR,2,2.0
2,3,Tyreek Hill,MIA,WR,3,3.0
3,4,Breece Hall,NYJ,RB,4,4.0
4,5,JaMarr Chase,CIN,WR,5,5.0


In [4]:
adp_df_cutoff.shape

(100, 6)

In [5]:
# find players of each position closest to ADP of 100

# initialize an empty dictionary
replacement_players = {
    'RB': '',
    'QB': '',
    'WR': '',
    'TE': ''
}

for _, row in adp_df_cutoff.iterrows():
    position = row['Pos']  # extract out the position and player value from each row as we loop through it
    player = row['Player']

    if position in replacement_players: # if the position is in the dict's keys
        replacement_players[position] = player # set that player as the replacement player

replacement_players

{'RB': 'Brian Robinson',
 'QB': 'Tua Tagovailoa',
 'WR': 'Rashee Rice',
 'TE': 'David Njoku'}

In [6]:
# replace player names with projected values

replacement_values = {} # initialize an empty dictionary

for position, player_name in replacement_players.items():
    player = df.loc[df['Player'] == player_name.strip()]
    replacement_values[position] = player['FantasyPoints'].tolist()[0]

replacement_values

{'RB': 165.8, 'QB': 262.1, 'WR': 161.2, 'TE': 149.2}

In [7]:
# function to grab each rows fantasy points and subtract replacement value

df['VOR'] = df.apply(
    lambda row: row['FantasyPoints'] - replacement_values.get(row['Pos']), axis=1
)

df.head()

,Player,Pos,Team,FantasyPoints,VOR
0,Jalen Hurts,QB,PHI,351.8,89.7
1,Josh Allen,QB,BUF,349.6,87.5
2,Patrick Mahomes,QB,KC,328.7,66.6
3,Lamar Jackson,QB,BAL,327.3,65.2
4,Christian McCaffrey,RB,SF,315.3,149.5


In [12]:
pd.set_option('display.max_rows', None) # turn off truncation of rows setting inherent to pandas

df['VOR Rank'] = df['VOR'].rank(ascending=False)

df = df.sort_values(by='VOR', ascending=False)

df.head()

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank
4,Christian McCaffrey,RB,SF,315.3,149.5,1.0
17,CeeDee Lamb,WR,DAL,275.5,114.3,2.0
16,Breece Hall,RB,NYJ,275.5,109.7,3.0
22,Bijan Robinson,RB,ATL,267.4,101.6,4.0
27,Tyreek Hill,WR,MIA,258.6,97.4,5.0


In [13]:
adp_df = adp_df.drop('Team', axis=1)

adp_df.head()

,Rank,Player,Pos,Current ADP,ADP Rank
0,1,Christian McCaffrey,RB,1,1.0
1,2,CeeDee Lamb,WR,2,2.0
2,3,Tyreek Hill,WR,3,3.0
3,4,Breece Hall,RB,4,4.0
4,5,JaMarr Chase,WR,5,5.0


In [14]:
# merge the two data sets
final_df = df.merge(adp_df, how='left', on=['Player', 'Pos'])

final_df.head()

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank
0,Christian McCaffrey,RB,SF,315.3,149.5,1.0,1.0,1.0,1.0
1,CeeDee Lamb,WR,DAL,275.5,114.3,2.0,2.0,2.0,2.0
2,Breece Hall,RB,NYJ,275.5,109.7,3.0,4.0,4.0,4.0
3,Bijan Robinson,RB,ATL,267.4,101.6,4.0,6.0,6.0,6.0
4,Tyreek Hill,WR,MIA,258.6,97.4,5.0,3.0,3.0,3.0


In [17]:
# calculate the difference between value rank and adp rank
final_df['Diff in ADP and VOR'] = final_df['ADP Rank'] - final_df['VOR Rank']

draft_pool = final_df.sort_values(by='ADP Rank', ascending=True)[:200]

draft_pool.head(20)

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
0,Christian McCaffrey,RB,SF,315.3,149.5,1.0,1.0,1.0,1.0,0.0
1,CeeDee Lamb,WR,DAL,275.5,114.3,2.0,2.0,2.0,2.0,0.0
4,Tyreek Hill,WR,MIA,258.6,97.4,5.0,3.0,3.0,3.0,-2.0
2,Breece Hall,RB,NYJ,275.5,109.7,3.0,4.0,4.0,4.0,1.0
11,JaMarr Chase,WR,CIN,239.6,78.4,12.0,5.0,5.0,5.0,-7.0
3,Bijan Robinson,RB,ATL,267.4,101.6,4.0,6.0,6.0,6.0,2.0
10,Justin Jefferson,WR,MIN,240.1,78.9,11.0,7.0,7.0,7.0,-4.0
9,AmonRa St Brown,WR,DET,243.7,82.5,10.0,8.0,8.0,8.0,-2.0
8,Jonathan Taylor,RB,IND,249.4,83.6,9.0,9.0,9.0,9.0,0.0
16,AJ Brown,WR,PHI,224.9,63.7,17.0,10.0,10.0,10.0,-7.0


In [ ]:
# Export the data to a CSV file

draft_pool.to_csv('Draft Values.csv', index=False)

In [28]:
rb_draft_pool = draft_pool.loc[draft_pool['Pos'] == 'RB']
qb_draft_pool = draft_pool.loc[draft_pool['Pos'] == 'QB']
wr_draft_pool = draft_pool.loc[draft_pool['Pos'] == 'WR']
te_draft_pool = draft_pool.loc[draft_pool['Pos'] == 'TE']

# top undervalued RBs
rb_draft_pool.sort_values(by='Diff in ADP and VOR', ascending=False)[:10]

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
119,JK Dobbins,RB,LAC,147.4,-18.4,120.0,170.0,171.0,171.0,51.0
148,Tyler Allgeier,RB,ATL,122.3,-43.5,149.0,190.0,190.0,190.0,41.0
156,Ty Chandler,RB,MIN,118.2,-47.6,157.0,185.0,187.0,187.0,30.0
146,Antonio Gibson,RB,NE,125.8,-40.0,147.0,164.0,167.0,167.0,20.0
27,James Cook,RB,BUF,208.6,42.8,28.0,42.0,42.0,42.0,14.0
20,Joe Mixon,RB,HOU,214.8,49.0,21.5,35.0,35.0,35.0,13.5
179,Jaleel McLaughlin,RB,DEN,102.9,-62.9,180.0,193.0,193.0,193.0,13.0
87,Austin Ekeler,RB,WAS,165.9,0.1,88.0,101.0,101.0,101.0,13.0
36,Alvin Kamara,RB,NO,202.7,36.9,37.0,49.0,49.0,49.0,12.0
173,Rico Dowdle,RB,DAL,106.8,-59.0,174.0,181.0,182.0,182.0,8.0


In [29]:
# top overvalued RBs
rb_draft_pool.sort_values(by='Diff in ADP and VOR', ascending=True)[:10]

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
268,AJ Dillon,RB,GB,54.2,-111.6,269.0,147.0,145.0,145.0,-124.0
280,Dameon Pierce,RB,HOU,49.9,-115.9,280.5,171.0,170.0,170.0,-110.5
230,Elijah Mitchell,RB,SF,75.3,-90.5,231.0,174.0,173.0,173.0,-58.0
165,Trey Benson,RB,ARI,113.4,-52.4,166.0,111.0,112.0,112.0,-54.0
212,Jaylen Wright,RB,MIA,85.4,-80.4,213.0,172.0,172.0,172.0,-41.0
168,Blake Corum,RB,LAR,112.4,-53.4,169.0,137.0,137.0,137.0,-32.0
86,Rhamondre Stevenson,RB,NE,166.4,0.6,87.0,59.0,59.0,59.0,-28.0
203,Khalil Herbert,RB,CHI,90.1,-75.7,204.0,180.0,177.0,177.0,-27.0
151,Jerome Ford,RB,CLE,119.0,-46.8,152.5,131.0,131.0,131.0,-21.5
115,Devin Singletary,RB,NYG,148.5,-17.3,116.0,95.0,95.0,95.0,-21.0


In [30]:
# top undervalued WRs
wr_draft_pool.sort_values(by='Diff in ADP and VOR', ascending=False)[:10]

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
107,Mike Williams,WR,NYJ,150.6,-10.6,108.0,186.0,184.0,184.0,76.0
114,Jerry Jeudy,WR,CLE,144.3,-16.9,115.0,178.0,180.0,180.0,65.0
124,Romeo Doubs,WR,GB,138.5,-22.7,125.0,188.0,186.0,186.0,61.0
132,Gabe Davis,WR,JAC,132.3,-28.9,133.0,191.0,189.0,189.0,56.0
142,Joshua Palmer,WR,LAC,124.6,-36.6,143.0,196.0,198.0,198.0,55.0
123,Jakobi Meyers,WR,LV,140.9,-20.3,124.0,173.0,175.0,175.0,51.0
128,Rashid Shaheed,WR,NO,135.7,-25.5,129.0,166.0,169.0,169.0,40.0
79,Christian Watson,WR,GB,163.7,2.5,80.0,120.0,120.0,120.0,40.0
154,Xavier Legette,WR,CAR,113.9,-47.3,155.0,189.0,191.0,191.0,36.0
118,Brian Thomas,WR,JAC,143.3,-17.9,119.0,153.0,153.0,153.0,34.0


In [31]:
# top overvalued WRs
wr_draft_pool.sort_values(by='Diff in ADP and VOR', ascending=True)[:10]

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
180,Odell Beckham,WR,MIA,97.7,-63.5,181.0,133.0,134.0,134.0,-47.0
223,Ricky Pearsall,WR,SF,76.5,-84.7,224.0,182.0,179.0,179.0,-45.0
120,Rome Odunze,WR,CHI,142.5,-18.7,121.0,99.0,99.0,99.0,-22.0
58,Jaylen Waddle,WR,MIA,181.3,20.1,58.5,37.0,37.0,37.0,-21.5
186,Dontayvion Wicks,WR,GB,94.1,-67.1,187.0,169.0,168.0,168.0,-19.0
44,Brandon Aiyuk,WR,SF,192.6,31.4,45.0,26.0,26.0,26.0,-19.0
93,Keenan Allen,WR,CHI,159.7,-1.5,94.0,76.0,76.0,76.0,-18.0
45,Drake London,WR,ATL,191.4,30.2,46.0,31.0,31.0,31.0,-15.0
92,Terry McLaurin,WR,WAS,159.8,-1.4,93.0,79.0,79.0,79.0,-14.0
98,Hollywood Brown,WR,KC,157.8,-3.4,99.0,85.0,85.0,85.0,-14.0


In [32]:
# top undervalued QBs
qb_draft_pool.sort_values(by='Diff in ADP and VOR', ascending=False)[:10]

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
66,Deshaun Watson,QB,CLE,273.6,11.5,66.5,176.0,178.0,178.0,111.5
74,Baker Mayfield,QB,TB,267.6,5.5,75.0,163.0,160.0,160.0,85.0
56,Trevor Lawrence,QB,JAC,282.5,20.4,57.0,136.0,135.0,135.0,78.0
69,Kirk Cousins,QB,ATL,273.2,11.1,70.0,143.0,143.0,143.0,73.0
60,Justin Herbert,QB,LAC,280.4,18.3,61.0,123.0,124.0,124.0,63.0
104,Geno Smith,QB,SEA,254.0,-8.1,105.0,162.0,162.0,162.0,57.0
59,Jayden Daniels,QB,WAS,281.7,19.6,60.0,114.0,114.0,114.0,54.0
64,Caleb Williams,QB,CHI,276.9,14.8,65.0,119.0,118.0,118.0,53.0
95,Aaron Rodgers,QB,NYJ,260.2,-1.9,96.0,145.0,146.0,146.0,50.0
97,Matthew Stafford,QB,LAR,258.9,-3.2,98.0,138.0,141.0,141.0,43.0


In [33]:
# top overvalued QBs
qb_draft_pool.sort_values(by='Diff in ADP and VOR', ascending=True)[:10]

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
322,Justin Fields,QB,PIT,62.3,-199.8,323.0,134.0,132.0,132.0,-191.0
175,Bo Nix,QB,DEN,201.7,-60.4,176.0,129.0,129.0,129.0,-47.0
153,JJ McCarthy,QB,MIN,215.0,-47.1,154.0,130.0,127.0,127.0,-27.0
152,Russell Wilson,QB,PIT,215.3,-46.8,152.5,127.0,128.0,128.0,-24.5
38,CJ Stroud,QB,HOU,296.8,34.7,39.0,41.0,41.0,41.0,2.0
37,Anthony Richardson,QB,IND,298.9,36.8,38.0,43.0,43.0,43.0,5.0
91,Tua Tagovailoa,QB,MIA,262.1,0.0,90.5,96.0,97.0,97.0,6.5
7,Josh Allen,QB,BUF,349.6,87.5,8.0,20.0,20.0,20.0,12.0
42,Kyler Murray,QB,ARI,293.6,31.5,43.5,56.0,56.0,56.0,12.5
14,Patrick Mahomes,QB,KC,328.7,66.6,15.0,28.0,28.0,28.0,13.0


In [34]:
# top undervalued TEs
te_draft_pool.sort_values(by='Diff in ADP and VOR', ascending=False)[:10]

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
163,Chigoziem Okonkwo,TE,TEN,97.3,-51.9,164.0,210.0,210.0,210.0,46.0
166,Hunter Henry,TE,NE,96.3,-52.9,167.0,206.0,206.0,206.0,39.0
172,Cade Otton,TE,TB,91.7,-57.5,173.0,212.0,212.0,212.0,39.0
149,Tyler Conklin,TE,NYJ,105.1,-44.1,150.0,187.0,183.0,183.0,33.0
176,Luke Musgrave,TE,GB,88.0,-61.2,177.0,203.0,200.0,200.0,23.0
138,Cole Kmet,TE,CHI,116.9,-32.3,138.5,149.0,148.0,148.0,9.5
147,Pat Freiermuth,TE,PIT,108.8,-40.4,148.0,156.0,156.0,156.0,8.0
196,Ben Sinnott,TE,WAS,76.3,-72.9,197.0,202.0,201.0,201.0,4.0
200,Noah Fant,TE,SEA,75.5,-73.7,200.0,200.0,204.0,204.0,4.0
50,Mark Andrews,TE,BAL,175.6,26.4,51.0,48.0,48.0,48.0,-3.0


In [35]:
# top overvalued TEs
te_draft_pool.sort_values(by='Diff in ADP and VOR', ascending=True)[:10]

,Player,Pos,Team,FantasyPoints,VOR,VOR Rank,Rank,Current ADP,ADP Rank,Diff in ADP and VOR
205,Dawson Knox,TE,BUF,71.9,-77.3,206.0,144.0,144.0,144.0,-62.0
100,Kyle Pitts,TE,ATL,144.0,-5.2,101.0,61.0,61.0,61.0,-40.0
96,Dalton Kincaid,TE,BUF,146.3,-2.9,97.0,57.0,57.0,57.0,-40.0
81,Trey McBride,TE,ARI,151.1,1.9,82.0,52.0,52.0,52.0,-30.0
102,Jake Ferguson,TE,DAL,143.2,-6.0,103.0,77.0,77.0,77.0,-26.0
48,Sam LaPorta,TE,DET,177.0,27.8,49.0,25.0,25.0,25.0,-24.0
77,George Kittle,TE,SF,153.6,4.4,78.0,54.0,54.0,54.0,-24.0
94,Evan Engram,TE,JAC,147.5,-1.7,95.0,72.0,72.0,72.0,-23.0
178,Isaiah Likely,TE,BAL,86.7,-62.5,179.0,159.0,161.0,161.0,-18.0
131,Brock Bowers,TE,LV,121.6,-27.6,132.0,116.0,116.0,116.0,-16.0
